In [2]:
import os
import sys
from datetime import datetime

import pandas as pd

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
karen_root = '/Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen'
output_root = '/Users/johannesnatterer/Developer/_output'

prior_file = 'Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv'
prior_dir = f'{karen_root}/Rock Music/Consolidated statements/old'
quarter_file = 'MOK 2026Q2_JN.xlsx'
quarter_dir = f'{karen_root}/Rock Music/Quarterly Statements/2026 Q2'
outputdirectory = output_root
outputfile = 'Rock_royalties_2021Q1_2026Q2_1_raw_combined.csv'
sheet_name = 'data'


def resolve_dir(rel):
    cwd = os.getcwd()
    alt = rel.replace('../../', '../', 1) if rel.startswith('../../') else rel
    candidates = [
        os.path.abspath(os.path.join(cwd, rel)),
        os.path.abspath(os.path.join(cwd, 'Karen_statements', rel)),
        os.path.abspath(os.path.join(cwd, alt)),
        os.path.abspath(os.path.join(os.path.dirname(cwd), rel)),
        os.path.abspath(os.path.join(cwd, '..', alt)),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f'Directory not found: {rel}\nTried:\n- ' + '\n- '.join(candidates))


def resolve_existing_file(name, *dirs):
    tried = []
    for folder in dirs:
        if not folder:
            continue
        path = os.path.join(folder, name)
        tried.append(path)
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f'File not found: {name}\nTried:\n- ' + '\n- '.join(tried))


prior_dir = resolve_dir(prior_dir)
outputdirectory = resolve_dir(outputdirectory)
try:
    quarter_dir = resolve_dir(quarter_dir)
except FileNotFoundError:
    quarter_dir = None

os.makedirs(outputdirectory, exist_ok=True)
output_path = os.path.join(outputdirectory, outputfile)
logfile_name = (
    os.path.splitext(outputfile)[0]
    + '_run_log_'
    + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '.txt'
)
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout


class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()

    def flush(self):
        for s in self.streams:
            s.flush()

    def isatty(self):
        return False

    def __getattr__(self, name):
        return getattr(self.streams[0], name)


sys.stdout = _Tee(_original_stdout, _log_file)
pd.options.display.float_format = '{:,.2f}'.format


def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f'Run log saved to: {log_path}')


def fmt_int(n):
    try:
        return f'{int(n):,}'
    except (TypeError, ValueError):
        return str(n)


def fmt_money(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def fmt_units(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def fmt_shape(df):
    return f'{fmt_int(df.shape[0])} rows × {len(df.columns)} columns'


def header(title):
    line = '=' * 72
    print(f'\n{line}\n  {title}\n{line}')


def subheader(title):
    print(f'\n--- {title} ---')


def type_name(t):
    return getattr(t, '__name__', str(t))


def column_type_summary(series):
    counts = series.map(type).value_counts()
    return ', '.join(f'{type_name(t)} {fmt_int(n)}' for t, n in counts.items())


def print_totals(label, df):
    share = df['SHARE AMOUNT (local FX)'].sum() if 'SHARE AMOUNT (local FX)' in df.columns else None
    units = df['UNIT'].sum() if 'UNIT' in df.columns else None
    royalty_hkd = df['Royalty (HKD)'].sum() if 'Royalty (HKD)' in df.columns else None
    line = f'  {label:<28} {fmt_shape(df)}'
    if share is not None:
        line += f'    SHARE {fmt_money(share):>16}'
    if units is not None:
        line += f'    UNIT {fmt_units(units):>18}'
    if royalty_hkd is not None and pd.notna(royalty_hkd):
        line += f'    Royalty (HKD) {fmt_money(royalty_hkd):>14}'
    print(line)


def align_columns(df1, df2):
    missing_in_df1 = df2.columns.difference(df1.columns)
    missing_in_df2 = df1.columns.difference(df2.columns)
    for col in missing_in_df1:
        df1[col] = pd.NA
    for col in missing_in_df2:
        df2[col] = pd.NA
    return df1, df2, list(missing_in_df1), list(missing_in_df2)


def convert_to_majority_type(df):
    df_copy = df.copy()
    conversions = []
    for col in df_copy.columns:
        type_counts = df_copy[col].map(type).value_counts()
        if type_counts.empty:
            continue
        majority_type = type_counts.idxmax()
        before = column_type_summary(df_copy[col])
        if majority_type == str:
            df_copy[col] = df_copy[col].astype(str)
        elif majority_type == float:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
        elif majority_type == int:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').astype('Int64')
        elif majority_type == bool:
            df_copy[col] = df_copy[col].astype(bool)
        else:
            df_copy[col] = df_copy[col].astype(str)
        after = column_type_summary(df_copy[col])
        if before != after:
            conversions.append((col, before, after))
    return df_copy, conversions


def mixed_type_columns(df):
    mixed = []
    for col in df.columns:
        types = set(df[col].dropna().map(type))
        if len(types) > 1:
            mixed.append((col, column_type_summary(df[col])))
    return mixed


header('Rock — combine quarterly files')
print(f'  Run started              : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Prior file               : {prior_file}')
print(f'  This quarter             : {quarter_file}')
print(f'  Output file              : {outputfile}')
print(f'  Output dir               : {outputdirectory}')
print(f'  Run log                  : {logfile_name}')

header('1. Required files')
try:
    path_prior = resolve_existing_file(prior_file, prior_dir)
    print(f'  [OK]       {path_prior}')
except FileNotFoundError as exc:
    print(f'  [MISSING]  {prior_file}')
    close_log()
    raise

try:
    path_quarter = resolve_existing_file(quarter_file, outputdirectory, quarter_dir)
    print(f'  [OK]       {path_quarter}')
except FileNotFoundError as exc:
    print(f'  [MISSING]  {quarter_file}')
    close_log()
    raise FileNotFoundError(str(exc))

header('2. Load files')
df_1 = pd.read_csv(path_prior, low_memory=False)
print_totals('Prior combined CSV', df_1)

df_2 = pd.read_excel(path_quarter, sheet_name=sheet_name)
print_totals(f'{quarter_file} / {sheet_name}', df_2)
drop_cols = [c for c in ['Royalty (HKD)'] if c in df_1.columns or c in df_2.columns]
for c in drop_cols:
    if c in df_1.columns:
        df_1 = df_1.drop(columns=c)
    if c in df_2.columns:
        df_2 = df_2.drop(columns=c)
    print(f'  Excluded from combine    : {c}')
df_2['CATALOG NO._MOD'] = 'mod.' + df_2['CATALOG NO.'].astype(str)
print('  Added                    : CATALOG NO._MOD  (= "mod." + CATALOG NO.)')

only_prior = sorted(set(df_1.columns) - set(df_2.columns))
only_quarter = sorted(set(df_2.columns) - set(df_1.columns))
if not only_prior and not only_quarter:
    print(f'  Columns                  : match ({len(df_1.columns)} columns)')
else:
    if only_prior:
        print(f'  Only in prior            : {only_prior}')
    if only_quarter:
        print(f'  Only in this quarter     : {only_quarter}')

header('3. Harmonise dtypes')
df_1c, conv_1 = convert_to_majority_type(df_1)
df_2c, conv_2 = convert_to_majority_type(df_2)
if conv_1:
    subheader('Prior CSV conversions')
    for col, before, after in conv_1:
        print(f'  {col:<24} {before}  →  {after}')
else:
    print('  Prior CSV                : no mixed-type conversions')
if conv_2:
    subheader('This quarter conversions')
    for col, before, after in conv_2:
        print(f'  {col:<24} {before}  →  {after}')
else:
    print('  This quarter             : no mixed-type conversions')

common_cols = df_1c.columns.intersection(df_2c.columns)
mismatches = []
for col in common_cols:
    types_df1 = set(df_1c[col].dropna().map(type))
    types_df2 = set(df_2c[col].dropna().map(type))
    if types_df1 != types_df2:
        mismatches.append((col, types_df1, types_df2))

if mismatches:
    print(f'  Type mismatches          : {fmt_int(len(mismatches))}')
    for col, t1, t2 in mismatches:
        print(
            f'    {col:<22} prior={sorted(type_name(t) for t in t1)}'
            f'    quarter={sorted(type_name(t) for t in t2)}'
        )
else:
    print('  Common column types      : match')

header('4. Combine')
df_1c, df_2c, added_to_prior, added_to_quarter = align_columns(df_1c, df_2c)
if added_to_prior:
    print(f'  Added empty cols to prior    : {added_to_prior}')
if added_to_quarter:
    print(f'  Added empty cols to quarter  : {added_to_quarter}')
if not added_to_prior and not added_to_quarter:
    print('  Column alignment          : none needed')

share_1 = df_1c['SHARE AMOUNT (local FX)'].sum()
share_2 = df_2c['SHARE AMOUNT (local FX)'].sum()
units_1 = df_1c['UNIT'].sum()
units_2 = df_2c['UNIT'].sum()

df_final = pd.concat([df_1c, df_2c], ignore_index=True)
df_final = df_final.sort_index(axis=1)
df_final = df_final.loc[:, ~df_final.columns.str.contains('^Unnamed')]

print_totals('Prior', df_1c)
print_totals('This quarter', df_2c)
print_totals('Combined', df_final)

share_diff = df_final['SHARE AMOUNT (local FX)'].sum() - (share_1 + share_2)
units_diff = df_final['UNIT'].sum() - (units_1 + units_2)
row_diff = df_final.shape[0] - (df_1c.shape[0] + df_2c.shape[0])
print()
print(f'  Row check                : prior {fmt_int(df_1c.shape[0])} + quarter {fmt_int(df_2c.shape[0])} = {fmt_int(df_1c.shape[0] + df_2c.shape[0])}    combined {fmt_int(df_final.shape[0])}    diff {fmt_int(row_diff)}')
print(f'  SHARE check              : diff {fmt_money(share_diff)}')
print(f'  UNIT check               : diff {fmt_units(units_diff)}')
if abs(share_diff) > 0.01 or abs(units_diff) > 0.01 or row_diff != 0:
    print('  ALERT: combined totals do not match prior + this quarter')
else:
    print('  Check                    : OK')

mixed_final = mixed_type_columns(df_final)
if mixed_final:
    print(f'  Mixed types in combined  : {fmt_int(len(mixed_final))}')
    for col, summary in mixed_final:
        print(f'    {col:<22} {summary}')
else:
    print('  Combined dtypes          : one type per column')

print()
print(f'  Output columns ({len(df_final.columns)}):')
print('    ' + ', '.join(df_final.columns.astype(str)))

header('5. Save output')
df_final.to_csv(output_path, index=False)
print(f'  CSV written              : {output_path}')
print(f'  {fmt_shape(df_final)}')
print(f'  SHARE AMOUNT (local FX)  : {fmt_money(df_final["SHARE AMOUNT (local FX)"].sum())}')
print(f'  UNIT                     : {fmt_units(df_final["UNIT"].sum())}')
print(f'  Run finished             : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
close_log()



  Rock — combine quarterly files
  Run started              : 2026-09-24 11:26:41
  Prior file               : Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv
  This quarter             : MOK 2026Q2_JN.xlsx
  Output file              : Rock_royalties_2021Q1_2026Q2_1_raw_combined.csv
  Output dir               : /Users/johannesnatterer/Developer/_output
  Run log                  : Rock_royalties_2021Q1_2026Q2_1_raw_combined_run_log_20260924_112641.txt

  1. Required files
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/Rock Music/Consolidated statements/old/Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/Rock Music/Quarterly Statements/2026 Q2/MOK 2026Q2_JN.xlsx

  2. Load files
  Prior combined CSV           1,398,905 rows × 29 columns    SHARE     6,877,194.97    UNIT   7,353,499,212.00
  MOK 202